In [1]:
from gentropy.common.session import Session
from gentropy.dataset.study_index import StudyIndex
from gentropy.dataset.study_locus import StudyLocus
from gentropy.dataset.variant_index import VariantIndex
from gentropy.method.drug_enrichment_from_evid import chemblDrugEnrichment
from pyspark.sql import functions as f

Loading BokehJS ...

/Users/yt4/Projects/Gentropy-manuscript/.venv/lib/python3.11/site-packages/pyspark/sql/pandas/functions.py:407: UserWarning:

In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.



In [2]:
session = Session(extended_spark_conf={"spark.driver.memory": "40G"})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/12/02 17:15:29 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/12/02 17:15:30 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


In [3]:
path_to_release_folder = "../../../data/25.06/"

sl = StudyLocus.from_parquet(session, path_to_release_folder + "output/credible_set")
si = StudyIndex.from_parquet(session, path_to_release_folder + "output/study")

disease_index_path = path_to_release_folder + "output/disease/disease.parquet"
disease_index_orig = session.spark.read.parquet(disease_index_path)

platform_chembl_evidence_path = path_to_release_folder + "output/evidence/sourceId=chembl"
chembl_evidence = session.spark.read.parquet(platform_chembl_evidence_path)
all_evidence = session.spark.read.parquet(path_to_release_folder + "output/evidence/")

efo_to_remove = chemblDrugEnrichment.selecting_all_decendands_based_on_efo_list(
    disease_index_orig=disease_index_orig,
    efo_ids="MONDO_0045024",
)

chembl_evidence.show(1)

l2g_full = session.spark.read.parquet("../../../data/intermediate_files/l2g_full_for_enrichment/")
l2g_full.count()

l2g_full.show(1)

g_p_s = session.spark.read.parquet("../../../data/intermediate_files/genes_therapeutic_areas")
g_p_s.count()

25/12/02 17:15:43 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+------------+---------------+-------------+-------------------+--------+----------+----+---------------------------+---------------------------+---------------------------------+--------------------------------+-----------------+-------------+----------+--------------------+--------+-------------+---------------------+--------------+-----------------+--------+----------------+---------------+----------+--------+-------------------+----------+----------------+--------------------+-------------------+-------------------------+-------------------------------------+-------------------------------------+--------------+----------+------------+-----------------+----------+----------------------------+-------------------+--------------+---------+--------------------------------+--------------------------------+--------------+--------------+--------+---------+----------+------------+-----------+--------------+-------------+----+------------------------+-----------------+-----------------------

8285

In [ ]:
l2g_full.count()

70400

In [ ]:
l2g_full = l2g_full.withColumn("diseaseId", f.explode("diseaseIds")).cache()
l2g_full.count()

77071

In [ ]:
l2g_full.select("geneId", "diseaseId").distinct().count()

36858

In [ ]:
l2g_full_df = l2g_full.select("geneId", "diseaseId").toPandas()

In [ ]:
l2g_full_df

,geneId,diseaseId
0,ENSG00000152954,MONDO_0004773
1,ENSG00000164307,MONDO_0004773
2,ENSG00000164307,MONDO_0004773
3,ENSG00000182095,MONDO_0004773
4,ENSG00000112062,MONDO_0004773
...,...,...
77066,ENSG00000115380,HP_0004872
77067,ENSG00000112297,HP_0004872
77068,ENSG00000115380,HP_0004872
77069,ENSG00000123219,HP_0004872


In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import fisher_exact

# assume l2g_full_df with columns ['geneId','diseaseId']
df = l2g_full_df.copy()
N = len(df)

# counts for each pair (a)
pair_counts = df.groupby(["geneId", "diseaseId"]).size().reset_index(name="a")

# marginal counts
gene_counts = df.groupby("geneId").size().reset_index(name="gene_total")
disease_counts = df.groupby("diseaseId").size().reset_index(name="disease_total")

# merge margins into pair table
tbl = pair_counts.merge(gene_counts, on="geneId", how="left").merge(disease_counts, on="diseaseId", how="left")

# compute 2x2 cells:
# a = same gene & same disease (already)
# b = same gene, different disease
# c = same disease, different gene
# d = neither same gene nor same disease
tbl["b"] = tbl["gene_total"] - tbl["a"]
tbl["c"] = tbl["disease_total"] - tbl["a"]
tbl["d"] = N - (tbl["a"] + tbl["b"] + tbl["c"])


# fisher exact per pair
def _fisher(row):
    table = [[int(row["a"]), int(row["b"])], [int(row["c"]), int(row["d"])]]
    try:
        orr, p = fisher_exact(table)  # two-sided by default
    except Exception:
        orr, p = (np.nan, np.nan)
    return pd.Series({"oddsratio": orr, "pvalue": p})


tbl[["oddsratio", "pvalue"]] = tbl.apply(_fisher, axis=1)

results_df = tbl[["geneId", "diseaseId", "a", "b", "c", "d", "oddsratio", "pvalue"]].copy()
results_df.reset_index(drop=True, inplace=True)

In [ ]:
from statsmodels.stats.multitest import multipletests

# run FDR (Benjamini-Hochberg) on the p-values and add adjusted p and reject flag
pvals = results_df["pvalue"].fillna(1.0).values
reject, pvals_adj, _, _ = multipletests(pvals, alpha=0.05, method="fdr_bh")

results_df["p_adj"] = pvals_adj

In [ ]:
(results_df["p_adj"] <= 0.001).sum()

np.int64(7221)

In [29]:
results_df[results_df["p_adj"] <= 0.0001]["geneId"].nunique()

2659

In [38]:
results_df_sign = results_df[results_df["p_adj"] <= 0.000001].copy()

In [39]:
from pyspark.sql.functions import broadcast

sig_pairs = results_df_sign[["geneId", "diseaseId"]].drop_duplicates()
sig_spark = session.spark.createDataFrame(sig_pairs)

In [40]:
l2g_full_filtered = l2g_full.join(sig_spark, on=["geneId", "diseaseId"], how="inner")

evidence = chemblDrugEnrichment.to_disease_target_evidence(
    table_with_score=l2g_full_filtered.drop("diseaseIds", "diseaseId"),
    score_column="score",
    datasource_id="l2g",
    study_locus=sl,
    study_index=si,
    min_score=0.1,
)

enrich = chemblDrugEnrichment.drug_enrichemnt_from_evidence(
    evid=evidence,
    disease_index_orig=disease_index_orig,
    chembl_orig=chembl_evidence,
    indirect_assoc_score_thr=0.1,
    efo_ancestors_to_remove=["MONDO_0045024"],
)

In [41]:
enrich

,clinicalPhase,odds_ratio,p_value,ci_low,ci_high,Relative success,ci_rs_low,ci_rs_high,rs_p_value,no_evid-low_clinphase,no_evid-high_clinphase,yes_evid-low_clinphase,yes_evid-high_clinphase,total_indirect_assoc
0,2+,1.825520,4.585068e-03,1.186144,2.809544,1.080796,1.035974,1.127557,3.238548e-04,6140,31002,23,212,17442
1,3+,2.526537,5.984441e-12,1.922598,3.320189,1.500184,1.371111,1.641409,9.915792e-19,20496,16646,77,158,17442
2,4+,4.700014,6.679966e-26,3.610160,6.118878,3.251498,2.765883,3.822373,2.622387e-46,32670,4472,143,92,17442
